# Fit Model on full Dev Set

- Use model config derrived for each of self/parent cohorts in exploration to fit a single model
- Extract coefs from each trained self/parent model

## Set Up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))

from src.config import BASE_PATH, SEED
from src.utils import col_config_sensitivity
from src.coef_visual import plot_heatmap
from src.residual_func import (
    build_model,
    get_saliency_map,
)

In [ ]:
BEST_PARAM_DICT = {
    "SQT": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Qwen3.6-27B",
            "desc_seg_type": "shot_zero",
            "alpha2": 15.0,
            "alpha3": 75.0,
            "temp": 0.0,
            "num_layers": 3,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg_type": "shot_few",
            "alpha2": 50.0,
            "alpha3": 15.0,
            "temp": 0.4,
            "num_layers": 3,
        },
    },
    "ST": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "15.0",
            "temp": 0.0,
            "num_layers": 2,
            "two_layer_base": "S",
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "25.0",
            "temp": 0.4,
            "num_layers": 2,
            "two_layer_base": "S",
        },
    },
    "T": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 1.0,
            "num_layers": 1,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 0.4,
            "num_layers": 1,
        },
    },
}

In [ ]:
shared_dict = {
    # "num_layers": 3,
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
parent_config = {
    "impact_type": "parent",
    "quest_cols": col_config_sensitivity["parent_quest"],
    **{
        k: v
        for k, v in BEST_PARAM_DICT["SQT"]["parent"].items()
        if k not in ["llm_model", "temp"]
    },
    **shared_dict,
}
self_config = {
    "impact_type": "self",
    "quest_cols": col_config_sensitivity["self_quest"],
    **{
        k: v
        for k, v in BEST_PARAM_DICT["SQT"]["self"].items()
        if k not in ["llm_model", "temp"]
    },
    **shared_dict,
}

## Build Model

In [ ]:
data_dir = BASE_PATH / "data"
model_parent, coef_parent, df_parent = build_model(
    parent_config, data_dir, temp=1.0, llm_model="Llama-3.3-70B-Instruct"
)
model_self, coef_self, df_self = build_model(
    self_config, data_dir, temp=0.4, llm_model="gemini-2.5-pro"
)

In [ ]:
coef_parent = coef_parent.rename(columns={"coef": "coef_parent"})
coef_self = coef_self.rename(columns={"coef": "coef_self"})
coef_combined = coef_parent.join(coef_self)
plot_heatmap(
    coef_combined,
    title="Learned Coef weights for final parent and self models",
    vmin=-1,
    vmax=1,
    xlabel="Cohort",
)

# Saliency Sanity Checks

In [ ]:
saliency_self = get_saliency_map(self_config, model_self, df_self, coef_self)

In [ ]:
saliency_parent = get_saliency_map(parent_config, model_parent, df_parent, coef_parent)

# LMG Feat Imp

- Randomly sample order of text outcomes n times
- Iteratively compute incremental r^2 after adding each outcome
- Avg across all 1000 samples to get avg contribution to accuracy

**TODO:**
- Improve parallelization

In [ ]:
from functools import reduce
from src.Regressor import ResidualRegressor
from src.utils import expand_text
from src.lmg_imp import get_perm_feat_imp
import pandas as pd

### Single Train-Test

In [ ]:
def single_train_test_lmg(config, n_repeats, df, starting_seed, scale=False):
    #### PREP DATA  ####
    # Get X/y train/test sets
    target_col = f"impact_score_{config['impact_type']}"
    df_expand, prob_outcomes = expand_text(
        df=df,
        agg_type=config["agg_type"],
        desc_seg_type=config["desc_seg_type"],
        impact_type=config["impact_type"],
    )
    # X
    X_train = df_expand.drop(target_col, axis=1)
    if scale:
        from sklearn.preprocessing import MinMaxScaler

        scaler = MinMaxScaler(clip=True, feature_range=(0, 1))
        X_train[prob_outcomes] = scaler.fit_transform(X_train[prob_outcomes])
    X_test = X_train
    # y
    y_train = df_expand[target_col].to_numpy()
    y_test = y_train

    #### RUN PERMS ####
    df_list = []
    for repeat in range(n_repeats):
        rand_state = starting_seed + repeat
        # Run feat imp
        lmg_df = get_perm_feat_imp(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            model_config_dict=config,
            random_state=rand_state,
            impact_type=config["impact_type"],
        )
        lmg_df = lmg_df.rename(
            columns={
                "original": f"original_{repeat}",
                "reversed": f"reversed_{repeat}",
            }
        )
        df_list.append(lmg_df)
    combined_df = reduce(lambda left, right: left.join(right, how="outer"), df_list)
    final_df = pd.DataFrame(
        {
            # avg for each outcome across all perms
            "avg_increase": combined_df.mean(axis=1).round(4),
            # stdev for each outcome across all perms
            "std_increase": combined_df.std(axis=1).round(4),
        }
    )
    return final_df

In [ ]:
parent_lmg = single_train_test_lmg(
    config=parent_config, n_repeats=500, df=df_parent, starting_seed=SEED, scale=True
)
self_lmg = single_train_test_lmg(
    config=self_config, n_repeats=500, df=df_self, starting_seed=SEED, scale=True
)

In [ ]:
combined_summary_single = pd.merge(
    parent_lmg["avg_increase"],
    self_lmg["avg_increase"],
    on="outcome",
    suffixes=("_parent", "_self"),
)
plot_heatmap(
    pivot=combined_summary_single,
    vmin=-0.05,
    vmax=0.05,
    title="Avg increase in % explained variance-- single TRAIN-TEST SPLIT",
    fmt=".2%",
    xlabel="Cohort",
)

In [ ]:
combined_summary_single_std = pd.merge(
    parent_lmg["std_increase"],
    self_lmg["std_increase"],
    on="outcome",
    suffixes=("_parent", "_self"),
)
plot_heatmap(
    pivot=combined_summary_single_std,
    vmin=-0.01,
    vmax=0.01,
    title="STDEV increase in % explained variance-- single TRAIN-TEST SPLIT",
    fmt=".2%",
    xlabel="Cohort",
)

### Via 10-repeated stratified CV

In [ ]:
from src.utils import get_df_path
from src.lmg_imp import lmg_imp_wrapper
from src.swarm import create_lmg_cmd, run_lmg_swarm

SAVE_DIR = BASE_PATH / "results/perm_res"
SWARM_DIR = BASE_PATH / "swarm_dir/lmg_imp"
CMD_DIR = SWARM_DIR / "cmds"
N_JOBS = 10
START_PERM = 0
END_PERM = 1000  # non-inclusive

In [ ]:
full_cmd_list = []
for impact_type in ["parent", "self"]:
    llm_model = (
        "Llama-3.3-70B-Instruct" if impact_type == "parent" else "gemini-2.5-pro"
    )
    temp = 0.6 if impact_type == "parent" else 0.4
    config_dict = parent_config if impact_type == "parent" else self_config
    df_path = get_df_path(
        df_dir=BASE_PATH / "data",
        llm_model=llm_model,
        agg_type=config_dict["agg_type"],
        desc_seg_type=config_dict["desc_seg_type"],
        impact_type=impact_type,
        temp=temp,
    )
    swarm_path = CMD_DIR / f"{impact_type}.swarm"
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    perm_cmds = []
    for perm_dir in (SAVE_DIR / "preds").iterdir():
        if perm_dir.name == "baseline":
            # NO baseline outcome ordering here
            continue
        perm_num = int(perm_dir.name.split("_")[1].strip(""))
        if (perm_num < START_PERM) or (perm_num >= END_PERM):
            continue
        cmd = create_lmg_cmd(
            config_dict=config_dict,
            n_cv_repeats=10,
            n_cv_splits=3,
            seed=SEED,
            n_lmg_samples=5000,
            df_path=df_path,
            n_jobs=N_JOBS,
            export_path=SAVE_DIR
            / "lmg_results"
            / perm_dir.name
            / f"{impact_type}.parquet",
            perm_dir=perm_dir,
        )
        perm_cmds.append(cmd)
    swarm_path.write_text("\n".join(perm_cmds))
    full_cmd_list += perm_cmds
print(f"Total commands: {len(full_cmd_list)}")

Sequentially locally

In [ ]:
# import subprocess
# for iteration, cmd in enumerate(cmd_list):
#     print(f"{iteration +1}/{len(cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

Run swarm

In [ ]:
from src.swarm import run_perm_swarm

run_perm_swarm(
    swarm_log_dir=SWARM_DIR / "logs",
    cmd_dir=CMD_DIR,
    scratch_gb=1,
    n_batch=100,
    process_gb=10,
    num_cpu=N_JOBS + 4,
    partition="quick",
    swarm_time="0:2:00",
    job_name="lmg_perm",
)

In [ ]:
def agg_over_perms(all_dfs):
    combined_df = reduce(lambda left, right: left.join(right, how="outer"), all_dfs)
    mean_cols = [col for col in combined_df.columns if "mean" in col]
    std_cols = [col for col in combined_df.columns if "std" in col]
    assert len(mean_cols) + len(std_cols) == combined_df.shape[1]
    final_df = pd.DataFrame(
        {
            # global mean
            "avg_mean": combined_df[mean_cols].mean(axis=1).round(4),
            # stdev of mean(avg across folds/repeats for a given permutation)
            "std_mean": combined_df[mean_cols].std(axis=1).round(4),
        }
    )
    return final_df

In [ ]:
global_dfs = {}
for impact_type in ["parent", "self"]:
    all_dfs = []
    for perm_dir in (SAVE_DIR / "lmg_results").iterdir():
        lmg_path = perm_dir / f"{impact_type}.parquet"
        if not lmg_path.exists():
            print(perm_dir.name)
            continue
        lmg_df = pd.read_parquet(lmg_path)
        summary_cols = [
            "original_mean",
            "reversed_mean",
            "original_std",
            "reversed_std",
        ]
        lmg_df_sub = lmg_df[summary_cols]
        lmg_df_sub = lmg_df_sub.rename(
            columns={
                og_name: f"{og_name}_{perm_dir.name.split('_')[1]}"
                for og_name in summary_cols
            }
        )
        all_dfs.append(lmg_df_sub)
    global_df = agg_over_perms(all_dfs)
    global_dfs[impact_type] = global_df

In [ ]:
combined_summary = pd.merge(
    global_dfs["parent"]["avg_mean"],
    global_dfs["self"]["avg_mean"],
    on="outcome",
    suffixes=("_parent", "_self"),
)
plot_heatmap(
    pivot=combined_summary,
    vmin=-0.01,
    vmax=0.01,
    title="Avg increase in % explained variance",
    fmt=".2%",
    xlabel="Cohort",
)

In [ ]:
combined_summary_std = pd.merge(
    global_dfs["parent"]["std_mean"],
    global_dfs["self"]["std_mean"],
    on="outcome",
    suffixes=("_parent", "_self"),
)
plot_heatmap(
    pivot=combined_summary_std,
    vmin=-0.025,
    vmax=0.025,
    title="Stdev of avg in increase in % explained variance",
    fmt=".2%",
    xlabel="Cohort",
)